# `TRG_LOC` ODI to Databricks Migration

**Conversion Timestamp:** 2024-07-30

**Description:** This notebook migrates data into the `trg_loc` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT CAST(${ODI_SESS_NO} AS BIGINT) AS odi_sess_no;

In [ ]:
display(spark.sql("""
  SELECT
    v_etl_job_type.etl_job_type,
    v_datasource_num_id.datasource_num_id,
    v_etl_proc_wid.etl_proc_wid,
    v_odi_sess_no.odi_sess_no
  FROM v_etl_job_type,
       v_datasource_num_id,
       v_etl_proc_wid,
       v_odi_sess_no
"""))

## Target Table: `TRG_LOC`

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: (No-op/Session marker)
-- SCEN_TASK_NO in {20}: (No-op/Session marker)
-- SCEN_TASK_NO in {30}: Insert data into TRG_LOC
INSERT INTO workspace.hr.trg_loc
  (
    location_id ,
    street_address ,
    postal_code ,
    city ,
    state_province ,
    country_id
  )
SELECT
  T1.location_id ,
  T1.street_address ,
  T1.postal_code ,
  T1.city ,
  T1.state_province ,
  T1.country_id
FROM
  workspace.hr.locations AS T1;

In [ ]:
%sql
SELECT COUNT(*) AS records_inserted
FROM workspace.hr.trg_loc;

## Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were created in this scenario to drop.

## Validation

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_loc
LIMIT 100;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema and table names have been converted to lowercase and prefixed with `workspace.`. For example, `HR.TRG_LOC` -> `workspace.hr.trg_loc`.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is Oracle-specific and not applicable to Databricks Delta Lake.
3.  **SCEN_TASK_NO:** The `SCEN_TASK_NO` markers have been preserved as comments within the relevant SQL cells.
4.  **Widget Parameters:** Placeholder widgets (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) and corresponding temporary views have been added as per the standard notebook structure, though they are not explicitly used in this specific simple `INSERT` statement.
5.  **DDL:** This conversion assumes that the target table `workspace.hr.trg_loc` already exists with a schema compatible with `workspace.hr.locations`. If DDL is required for `trg_loc`, it should be added manually.